**Week 11**<br />Wednesday, Nov. 11

# Language Modeling, Continued

LX 394/694<br />
Coding for Linguists

Sophie Hao


## Announcements

- HW 3 is released!
- Due **Monday, Nov. 23** at 9 AM (Week 13)
- Train and evaluate a spam classifier
- HW 2 late submissions accepted until Monday, Nov. 16 (Week 12)

## Recap: Language Modeling

**Language modeling** is an NLP task where, given a sequence of tokens $x = x_1x_2 \dots x_l$ where $x_1 = \texttt{'<s>'}$, the **language model** must predict the next token, $y$.

Given input $x$, the language model's output is:

$$
\hat{y} = \begin{Bmatrix}
   \text{the:} & \mathbb{P}[\text{the} | x] \\
   \text{of:} & \mathbb{P}[\text{of} | x] \\
   \text{in:} & \mathbb{P}[\text{in} | x] \\
   \vdots & \vdots
\end{Bmatrix}
$$


## Recap: N-Gram Language Models

Like unigram and bigram POS taggers, there are also unigram and bigram language models.

**Unigram Language Model:**
$$\mathbb{P}[w | x] = \mathbb{P}[w]$$

**Bigram Language Model:**
$$\mathbb{P}[w | x] = \mathbb{P}[w | x_l]$$

**Trigram Language Model:**
$$\mathbb{P}[w | x] = \mathbb{P}[w | x_{l - 1}x_l]$$

## Recap: Training an N-Gram Language Model

Let $c$ be a training corpus.

**Unigram Model:** ($n = 1$)

$$\mathbb{P}[w] = \frac{c\texttt{.count}(w)}{\texttt{len}(c)}$$

**Bigram Model:** ($n = 2$)

$$\mathbb{P}[w|x_l] = \frac{c\texttt{.count}(x_l w)}{c\texttt{.count}(x_l)}$$

**Trigram Model:** ($n = 3$)

$$\mathbb{P}[w|x_{l - 1}x_l] = \frac{c\texttt{.count}(x_{l - 1} x_l w)}{c\texttt{.count}(x_{l - 1}x_l)}$$

## Recap: Language Modeling in NLTK

In [1]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE


def train_lm(n: int, corpus) -> MLE:
    """ Train an n-gram model on corpus. """
    training_data, vocab = padded_everygram_pipeline(n, corpus)
    lm = MLE(n)
    lm.fit(training_data, vocab)
    return lm

## Generating with a Prompt

In [2]:
import nltk
from nltk.corpus import brown

# Train a trigram model
brown_lm = train_lm(3, brown.sents())

In [111]:
# Predict the next token
prompt_raw = "Boston University is located in the Commonwealth of"
prompt_tokens = nltk.word_tokenize(prompt_raw)
brown_lm.generate(10, text_seed=prompt_tokens)

['Puerto',
 'Rico',
 ',',
 'and',
 'promote',
 'fundamental',
 'scientific',
 'research',
 'and',
 'study']

## Exercise: Probability of a Text

What is the probability that our `brown_lm` trigram model will generate:

```python
"Boston University is located in the state of Massachusetts."
``` 

non-greedily?

## Probability of a Text

The probability that the sequence $y = y_1 y_2 \dots y_l$ will occur after $x$ is given by:

$$\begin{align*}
\mathbb{P}[y | x] &= \mathbb{P}[y_1 | x] \mathbb{P}[y_2| xy_1]  \dots \mathbb{P}[y_l |xy_1y_2 \dots y_{l - 1}] \\
&= \prod_{i = 1}^{l} \mathbb{P}[y_i | xy_1 y_2 \dots y_{i - 1}]
\end{align*}$$

Let's try computing the probability of a text.

In [112]:
# Prepare a text
text_raw = "The city of Boston is located in the state of Massachusetts."
text_tokens = ["<s>", "<s>"] + nltk.word_tokenize(text_raw) + ["</s>"]

# Multiply probabilities together
p = 1.
for w1, w2, w3 in nltk.trigrams(text_tokens):
    p *= brown_lm.score(w3, context=(w1, w2))
    
p

0.0

## Underflow

In a computer, `float`s are represented using **binary scientific notation**. For example:

$$
\begin{align*}
    1 &= 1 \times 2^0 \\
    2 &= 1 \times 2^1 \\
    3 &= 1.1 \times 2^1 \\
    3.14 &\approx 1.10010010 \times 2^1 \\
    .01 &\approx 1.10011001 \times 2^{-7}
\end{align*}
$$

If the exponent is too low, then the number will be **rounded to 0**.

## Minimum Exponent in Python

Look for `max_exp` and `min_exp`:

In [113]:
import sys

sys.float_info

sys.float_info(max=1.7976931348623157e+308, max_exp=1024, max_10_exp=308, min=2.2250738585072014e-308, min_exp=-1021, min_10_exp=-307, dig=15, mant_dig=53, epsilon=2.220446049250313e-16, radix=2, rounds=1)

## Log Probabilities

To prevent underflow, we usually work with **log probabilities**. NLTK uses base 2.

$$\log_2(ab) = \log_2(a) + \log_2(b)$$

$$
\log_2(\mathbb{P}[y | x]) = \sum_{i = 1}^{l} \log_2(\mathbb{P}[y_i | xy_1 y_2 \dots y_{i - 1}])
$$

## Probability of a Text, Attempt 2

This time, let's use **log probabilities**.

In [114]:
# Prepare a text
text_raw = "The city of Boston is located in the state of Massachusetts."
text_tokens = ["<s>", "<s>"] + nltk.word_tokenize(text_raw) + ["</s>"]

# Add log probabilities together
p = 0.
for w1, w2, w3 in nltk.trigrams(text_tokens):
    p += brown_lm.logscore(w3, context=(w1, w2))
    
p

-inf

## Error Analysis

Let's look at each of the token probabilities separately.

In [115]:
from tabulate import tabulate  # To install: pip install tabulate

data = [("Token", "Context", "Prob", "Log Prob")]
x = "The city of Boston is located in the state of Massachusetts."
tokens = ["<s>", "<s>"] + nltk.word_tokenize(x) + ["</s>"]
for w1, w2, w3 in nltk.trigrams(tokens):
    p = brown_lm.logscore(w3, context=(w1, w2))
    data.append((w3, (w1, w2), 2 ** p, p))
print(tabulate(data, headers="firstrow"))

Token          Context                        Prob    Log Prob
-------------  -----------------------  ----------  ----------
The            ('<s>', '<s>')           0.114126      -3.1313
city           ('<s>', 'The')           0.00122249    -9.67596
of             ('The', 'city')          0           -inf
Boston         ('city', 'of')           0           -inf
is             ('of', 'Boston')         0           -inf
located        ('Boston', 'is')         0           -inf
in             ('is', 'located')        0.0769231     -3.70044
the            ('located', 'in')        0.466667      -1.09954
state          ('in', 'the')            0.00432744    -7.85227
of             ('the', 'state')         0.150838      -2.72893
Massachusetts  ('state', 'of')          0.0121951     -6.35755
.              ('of', 'Massachusetts')  0.25          -2
</s>           ('Massachusetts', '.')   1              0


## Data Sparsity

In [116]:
# 0 probability means the n-gram is not in the training corpus
brown_lm.context_counts(("The", "city"))

FreqDist({'has': 2, 'paid': 1, 'expects': 1, 'is': 1, 'sewer': 1, 'was': 1, 'had': 1})

In [117]:
brown_lm.context_counts(("The", "City"))

FreqDist({'of': 2, 'Purchasing': 1, 'Planning': 1, 'Government': 1})

In [118]:
# Sometimes the context is not in the training corpus either
brown_lm.context_counts(("green", "ideas"))

FreqDist({})

In [119]:
# Sometimes the tokens aren't even in the training corpus
brown_lm.context_counts(("Romi", "Hill"))

FreqDist({})

## Exercise: Missing Contexts

Recall that for a trigram model,
$$\mathbb{P}[w \mathrel{|} \text{green ideas}] = \frac{c\text{.count}(\text{green ideas $w$})}{c\text{.count}(\text{green ideas})}$$
where $c$ is the training corpus.

If $c$ does not contain `('green', 'ideas')`, then the right-hand side is $\frac{0}{0}$! 

How do we avoid dividing by 0? 

**Hint:** Think about HW 1, as well as POS tagging

## Add-k Smoothing

When we do add-$k$ smoothing for an $n$-gram language model, we pretend that $c$ contains $k$ additional instances of each possible $n$-gram.

**Unigram Language Model:** ($n = 1$)
$$\mathbb{P}[w] = \frac{c\text{.count}(w) + k}{\text{len}(c) + k \cdot \text{vocab_size}}$$

**Bigram Language Model:** ($n = 2$)
$$\mathbb{P}[w \mathrel{|} x_l] = \frac{c\text{.count}(x_lw) + k}{c\text{.count}(x_l) + k \cdot \text{vocab_size}^2}$$

**Trigram Language Model:** ($n = 3$)
$$\mathbb{P}[w \mathrel{|} x_{l - 1}x_l] = \frac{c\text{.count}(x_{l - 1}x_lw) + k}{c\text{.count}(x_{l - 1}x_l) + k \cdot \text{vocab_size}^3}$$

**N-Gram Language Model:** ($n > 1$)
$$\mathbb{P}[w \mathrel{|} x_{l - n + 2}x_{l - n + 3} \dots x_l] = \frac{c\text{.count}(x_{l - n + 2}x_{l - n + 3} \dots x_lw) + k}{c\text{.count}(x_{l - n + 2}x_{l - n + 3} \dots x_l) + k \cdot \text{vocab_size}^n}$$

## OOV Tokens

**Out-of-vocabulary** (OOV) tokens are represented using the special token `'<UNK>'`.

In [120]:
brown_lm.vocab.lookup("Romi")

'<UNK>'

## Next Time

Language Modeling, Continued